In [1]:
import pandas as pd
import numpy as np
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data = load_all_data()
pl   = data['profitandloss']
bs   = data['balancesheet']

# Merge P&L and Balance Sheet
merged = pd.merge(
    pl,
    bs[['company_id', 'year', 'equity_capital', 'reserves',
        'borrowings', 'total_assets', 'fixed_assets', 'investments']],
    on=['company_id', 'year'],
    how='inner'
)

merged['total_equity'] = merged['equity_capital'] + merged['reserves']

print(f"Merged rows: {len(merged)}")
print("Ready!")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Merged rows: 1082
Ready!


In [2]:
def compute_de_ratio(df):
    """
    D/E = Borrowings / (Equity Capital + Reserves)
    0 = debt free
    None = negative equity
    """
    df = df.copy()
    df['debt_to_equity'] = np.where(
        df['total_equity'] > 0,
        (df['borrowings'] / df['total_equity']).round(2),
        None
    )
    return df

merged = compute_de_ratio(merged)

# Show some interesting results
print("Debt Free companies (D/E = 0) in latest year:")
latest = merged[merged['year'] == '2024-03']
debt_free = latest[latest['debt_to_equity'] == 0]
print(f"  Count: {len(debt_free)}")
print(f"  Examples: {list(debt_free['company_id'][:5])}")

print("\nHighly leveraged companies (D/E > 2):")
high_de = latest[latest['debt_to_equity'] > 2]
print(high_de[['company_id', 'debt_to_equity']].sort_values(
    'debt_to_equity', ascending=False).head(5).to_string(index=False))

Debt Free companies (D/E = 0) in latest year:
  Count: 9
  Examples: ['BAJAJHLDNG', 'BOSCHLTD', 'DIVISLAB', 'ICICIGI', 'ITC']

Highly leveraged companies (D/E > 2):
company_id debt_to_equity
     CANBK          14.87
 UNIONBANK          12.82
BANKBARODA          12.14
       PFC           8.52
      IRFC           8.38


In [3]:
def compute_icr(df):
    """
    ICR = (Operating Profit + Other Income) / Interest
    None if interest = 0 (debt free)
    """
    df = df.copy()
    df['interest_coverage'] = np.where(
        df['interest'] > 0,
        ((df['operating_profit'] + df['other_income']) /
         df['interest']).round(2),
        None
    )
    return df

merged = compute_icr(merged)

# Show dangerous ICR < 1 companies
print("Companies with ICR < 1 (danger zone) in latest year:")
latest = merged[merged['year'] == '2024-03']
danger = latest[
    latest['interest_coverage'].notna() &
    (latest['interest_coverage'] < 1)
]
if len(danger) > 0:
    print(danger[['company_id', 'interest_coverage']].to_string(index=False))
else:
    print("  None! All companies can cover their interest. ✅")

print("\nStrong ICR > 10 companies:")
strong = latest[
    latest['interest_coverage'].notna() &
    (latest['interest_coverage'] > 10)
]
print(f"  Count: {len(strong)}")
print(f"  Examples: {list(strong['company_id'][:5])}")

Companies with ICR < 1 (danger zone) in latest year:
company_id interest_coverage
       PFC              0.58
SHRIRAMFIN              0.68
 UNIONBANK              0.35

Strong ICR > 10 companies:
  Count: 50
  Examples: ['ABB', 'AMBUJACEM', 'ASIANPAINT', 'ATGL', 'BAJAJ-AUTO']


In [4]:
def compute_net_debt(df):
    """
    Net Debt = Borrowings - Investments
    Negative = net cash positive (good sign!)
    """
    df = df.copy()
    df['net_debt'] = (df['borrowings'] - df['investments']).round(2)
    return df

merged = compute_net_debt(merged)

# Show net cash positive companies
print("Net Cash Positive companies in latest year (Net Debt < 0):")
latest = merged[merged['year'] == '2024-03']
net_cash = latest[latest['net_debt'] < 0]
print(f"  Count: {len(net_cash)}")
print(net_cash[['company_id', 'net_debt']].sort_values(
    'net_debt').head(5).to_string(index=False))

Net Cash Positive companies in latest year (Net Debt < 0):
  Count: 37
company_id  net_debt
      LICI  -4976133
   SBILIFE   -384943
  HDFCLIFE   -290178
ICICIPRULI   -288528
    JIOFIN   -133292


In [6]:
def compute_asset_turnover(df):
    """
    Asset Turnover = Sales / Total Assets
    Higher = more efficient use of assets
    """
    df = df.copy()
    df['asset_turnover'] = np.where(
        df['total_assets'] > 0,
        (df['sales'] / df['total_assets']).round(2),
        np.nan  # use np.nan instead of None
    )
    df['asset_turnover'] = pd.to_numeric(df['asset_turnover'], errors='coerce')
    return df

def compute_fixed_asset_turnover(df):
    """
    Fixed Asset Turnover = Sales / Fixed Assets
    None if fixed assets = 0 (pure service companies)
    """
    df = df.copy()
    df['fixed_asset_turnover'] = np.where(
        df['fixed_assets'] > 0,
        (df['sales'] / df['fixed_assets']).round(2),
        np.nan  # use np.nan instead of None
    )
    df['fixed_asset_turnover'] = pd.to_numeric(
        df['fixed_asset_turnover'], errors='coerce'
    )
    return df

merged = compute_asset_turnover(merged)
merged = compute_fixed_asset_turnover(merged)

print("Asset Turnover — Top 5 most efficient companies:")
latest = merged[merged['year'] == '2024-03']
print(latest.nlargest(5, 'asset_turnover')[
    ['company_id', 'asset_turnover', 'fixed_asset_turnover']
].to_string(index=False))

Asset Turnover — Top 5 most efficient companies:
company_id  asset_turnover  fixed_asset_turnover
       BEL          125.89               1126.00
       HAL           67.51                126.59
    INDIGO           56.39                125.74
        LT            8.03                 44.39
     DMART            2.40                  3.79


In [7]:
print("Leverage & Efficiency Ratios — Coverage Summary:")
print(f"  Total rows:            {len(merged)}")
print(f"  D/E computed:          {merged['debt_to_equity'].notna().sum()}")
print(f"  ICR computed:          {merged['interest_coverage'].notna().sum()}")
print(f"  Net Debt computed:     {merged['net_debt'].notna().sum()}")
print(f"  Asset Turnover:        {merged['asset_turnover'].notna().sum()}")
print(f"  Fixed Asset Turnover:  {merged['fixed_asset_turnover'].notna().sum()}")

print("\nSample — TCS latest year:")
tcs = merged[merged['company_id'] == 'TCS'].tail(1)[[
    'company_id', 'year', 'debt_to_equity',
    'interest_coverage', 'net_debt', 'asset_turnover'
]]
print(tcs.to_string(index=False))

print("\nSample — RELIANCE latest year:")
rel = merged[merged['company_id'] == 'RELIANCE'].tail(1)[[
    'company_id', 'year', 'debt_to_equity',
    'interest_coverage', 'net_debt', 'asset_turnover'
]]
print(rel.to_string(index=False))

Leverage & Efficiency Ratios — Coverage Summary:
  Total rows:            1082
  D/E computed:          1082
  ICR computed:          1033
  Net Debt computed:     1082
  Asset Turnover:        1081
  Fixed Asset Turnover:  1081

Sample — TCS latest year:
company_id    year debt_to_equity interest_coverage  net_debt  asset_turnover
       TCS 2024-03           0.09              87.1    -23741            1.66

Sample — RELIANCE latest year:
company_id    year debt_to_equity interest_coverage  net_debt  asset_turnover
  RELIANCE 2024-03           0.58              7.73    233319            0.51
